# CEREBRO — Reactive 3D Brain Visualization

TRIBE v2-style interactive brain surface viewer.
Given any of the 200 THINGS-EEG2 test images, shows the **actual** and **predicted** EEG response
projected onto an inflated 3D cortical surface via MNE sLORETA source localization.

**Prerequisites:** Google Drive mounted with all Phase 3–5 artifacts present.

## 0 — Setup

In [ ]:
!pip install -q mne nilearn plotly ipywidgets Pillow
!pip install -q torch torchvision transformers==4.44.0 numpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json

ROOT       = '/content/drive/MyDrive/tribe-eeg'      # Bala's data (read-only shortcut)
WRITE_ROOT = '/content/drive/MyDrive/tribe-eeg-viz'  # your own writable folder

CKPT_DIR  = f'{ROOT}/checkpoints/multi_subject_clip_seed42'   # 63-ch full model
EMB_DIR   = f'{ROOT}/embeddings'
PROC_DIR  = f'{ROOT}/processed'
IMG_ZIP   = f'{ROOT}/test_images.zip'
META_FILE = f'{PROC_DIR}/eeg_meta.json'

VIZ_CACHE        = f'{WRITE_ROOT}/viz_cache'    # pre-computed source maps
MNE_SUBJECTS_DIR = f'{WRITE_ROOT}/mne_subjects' # fsaverage download
os.makedirs(VIZ_CACHE, exist_ok=True)
os.makedirs(MNE_SUBJECTS_DIR, exist_ok=True)

print('Paths OK')

## 1 — Load artifacts

In [ ]:
import numpy as np

with open(META_FILE) as f:
    meta = json.load(f)

ch_names_all = meta['ch_names']           # 64 names including 'stim'
stim_idx     = ch_names_all.index('stim')
eeg_ch_names = [c for c in ch_names_all if c != 'stim']  # 63 EEG channels
times_ms     = np.array(meta['times_ms'])  # (100,) ms axis
n_ch, n_t    = len(eeg_ch_names), len(times_ms)   # 63, 100

print(f'{n_ch} EEG channels, {n_t} timepoints ({times_ms[0]:.0f} to {times_ms[-1]:.0f} ms)')

In [ ]:
# Load averaged test EEG for all 10 subjects
# Inspect keys on first file so we don't hardcode the wrong name

n_subjects = 10
actual_eeg = []

_probe = np.load(f'{PROC_DIR}/eeg_test_avg_sub-01.npz')
print('Keys in npz:', list(_probe.files))
eeg_key = [k for k in _probe.files if k not in ('ch_names', 'times')][0]
print('Using EEG key:', eeg_key, '  shape:', _probe[eeg_key].shape)

for sub in range(1, n_subjects + 1):
    path = f'{PROC_DIR}/eeg_test_avg_sub-{sub:02d}.npz'
    d = np.load(path, allow_pickle=True)
    eeg = d[eeg_key]                           # (200, 64, 100)
    eeg = np.delete(eeg, stim_idx, axis=1)     # (200, 63, 100)
    actual_eeg.append(eeg.astype(np.float32))

actual_eeg = np.stack(actual_eeg)              # (10, 200, 63, 100)
grand_avg_actual = actual_eeg.mean(axis=0)     # (200, 63, 100)
print('Actual EEG:', actual_eeg.shape, '  grand avg:', grand_avg_actual.shape)

In [ ]:
# Load CLIP ViT-L/14 test embeddings: (200, 768)
clip_test = np.load(f'{EMB_DIR}/clip_vitl14_test.npy').astype(np.float32)
print('CLIP test embeddings:', clip_test.shape)

## 2 — Load model and run inference

In [ ]:
import torch
import torch.nn as nn

# Exact architecture from CEREBRO.ipynb Phase 5

class TransformerEncoder(nn.Module):
    def __init__(self, D_emb, d_model, n_heads, n_layers, ffn, n_tokens):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(D_emb, d_model), nn.LayerNorm(d_model))
        self.pos  = nn.Parameter(torch.zeros(1, n_tokens, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ffn,
            dropout=0.1, activation='gelu', batch_first=True, norm_first=True,
        )
        self.enc  = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):          # x: (B, n_tokens, D_emb)
        h = self.stem(x) + self.pos
        h = self.enc(h)
        return self.norm(h)[:, 0]  # position-0 readout


class MultiSubjectTransformer(nn.Module):
    def __init__(self, D_emb, CT, d_model, n_heads, n_layers, ffn, n_tokens, n_subjects):
        super().__init__()
        self.encoder    = TransformerEncoder(D_emb, d_model, n_heads, n_layers, ffn, n_tokens)
        self.n_subjects = n_subjects
        self.W = nn.Parameter(torch.empty(n_subjects + 1, d_model, CT))
        nn.init.trunc_normal_(self.W, std=0.02)
        self.b = nn.Parameter(torch.zeros(n_subjects + 1, CT))

    def forward(self, x, subj_id):  # x: (B, n_tokens, D), subj_id: (B,) long
        h = self.encoder(x)          # (B, d_model)
        W = self.W[subj_id]          # (B, d_model, CT)
        b = self.b[subj_id]          # (B, CT)
        return torch.einsum('bd,bdc->bc', h, W) + b  # (B, CT)


def build_context(emb_array, K=10):
    """Build RSVP context window. Returns (N, 1+K, D)."""
    N, D = emb_array.shape
    K_left  = K // 2
    K_right = K - K_left
    pad = np.zeros((N + K, D), dtype=emb_array.dtype)
    pad[K_left:K_left + N] = emb_array
    out = np.zeros((N, 1 + K, D), dtype=emb_array.dtype)
    for i in range(N):
        out[i, 0]           = emb_array[i]
        out[i, 1:1+K_left]  = pad[i : i+K_left]
        out[i, 1+K_left:1+K] = pad[i+K_left+1 : i+K_left+1+K_right]
    return out

print('Model classes defined.')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

D_emb, d_model, n_heads, n_layers, ffn = 768, 384, 6, 4, 1536
K, n_tokens = 10, 11
CT = n_ch * n_t  # 63 * 100 = 6300

model = MultiSubjectTransformer(
    D_emb=D_emb, CT=CT, d_model=d_model,
    n_heads=n_heads, n_layers=n_layers, ffn=ffn,
    n_tokens=n_tokens, n_subjects=n_subjects
).to(device)

ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f"Checkpoint loaded  (val R = {ckpt.get('val_r', 'n/a'):.4f})")

In [ ]:
# Run inference for all 200 test images × all 10 subjects
# Output: pred_eeg shape (10, 200, 63, 100)

ctx = build_context(clip_test, K=K)  # (200, 11, 768)
ctx_t = torch.tensor(ctx, dtype=torch.float32).to(device)

pred_eeg = np.zeros((n_subjects, 200, n_ch, n_t), dtype=np.float32)

with torch.no_grad():
    for sub_idx in range(n_subjects):
        subj_ids = torch.full((200,), sub_idx, dtype=torch.long, device=device)
        out = model(ctx_t, subj_ids)           # (200, CT)
        pred_eeg[sub_idx] = out.cpu().numpy().reshape(200, n_ch, n_t)
        print(f'  subject {sub_idx+1}/10 done', flush=True)

grand_avg_pred = pred_eeg.mean(axis=0)  # (200, 63, 100)
print('Predicted EEG:', pred_eeg.shape, '  grand avg:', grand_avg_pred.shape)

In [ ]:
# Quick sanity: Pearson R between predicted and actual grand averages
from scipy.stats import pearsonr

rs = []
for i in range(200):
    r, _ = pearsonr(grand_avg_pred[i].ravel(), grand_avg_actual[i].ravel())
    rs.append(r)
print(f'Grand-avg Pearson R across 200 test images: {np.mean(rs):.4f} ± {np.std(rs):.4f}')

## 3 — MNE source localization setup

One-time setup (~5–10 min). Results cached to Drive so subsequent runs skip this.

In [ ]:
import mne
mne.set_log_level('WARNING')

sfreq   = 100.0
info    = mne.create_info(ch_names=eeg_ch_names, sfreq=sfreq, ch_types='eeg')
montage = mne.channels.make_standard_montage('biosemi64')

montage_ch = set(montage.ch_names)
missing = [c for c in eeg_ch_names if c not in montage_ch]
if missing:
    print('Channels not in BioSemi64 montage (dropping):', missing)
    keep     = [c for c in eeg_ch_names if c in montage_ch]
    info     = mne.create_info(ch_names=keep, sfreq=sfreq, ch_types='eeg')
    keep_idx = [eeg_ch_names.index(c) for c in keep]
else:
    keep     = eeg_ch_names
    keep_idx = list(range(n_ch))

info.set_montage(montage)

# Add average EEG reference projector via a dummy RawArray (works across MNE versions)
import numpy as np
raw_tmp = mne.io.RawArray(np.zeros((len(keep), 2)), info, verbose=False)
raw_tmp.set_eeg_reference('average', projection=True, verbose=False)
info = raw_tmp.info

print(f'MNE Info: {len(keep)} EEG channels at {sfreq} Hz, avg-ref proj added')

In [ ]:
# Download fsaverage (MNE standard brain, ~1.5 GB, cached on Drive)
import os
os.environ['MNE_DATA'] = MNE_SUBJECTS_DIR

fs_dir = mne.datasets.fetch_fsaverage(subjects_dir=MNE_SUBJECTS_DIR, verbose=True)
subjects_dir = MNE_SUBJECTS_DIR
print('fsaverage at:', fs_dir)

In [ ]:
FWD_FILE = f'{VIZ_CACHE}/fsaverage-fwd.fif'

if os.path.exists(FWD_FILE):
    fwd = mne.read_forward_solution(FWD_FILE)
    print('Loaded cached forward solution.')
else:
    # Source space: oct5 = 4098 sources/hemisphere = 8196 total
    src = mne.setup_source_space(
        'fsaverage', spacing='oct5', subjects_dir=subjects_dir, add_dist=False
    )
    # BEM (boundary element model) — use pre-computed shell from fsaverage
    bem = mne.read_bem_solution(
        os.path.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-5120-5120-5120-bem-sol.fif')
    )
    # Forward solution
    fwd = mne.make_forward_solution(
        info, trans='fsaverage', src=src, bem=bem,
        meg=False, eeg=True, mindist=5.0, n_jobs=-1
    )
    mne.write_forward_solution(FWD_FILE, fwd, overwrite=True)
    print('Forward solution computed and cached.')

print(fwd)

In [ ]:
import os

INV_FILE = f'{VIZ_CACHE}/fsaverage-inv.fif'

# Noise covariance from pre-stimulus baseline (-200 to 0 ms = timepoints 0..19)
baseline_data = grand_avg_actual[:, keep_idx, :20]  # (200, n_ch, 20)
baseline_flat = baseline_data.transpose(1, 0, 2).reshape(len(keep_idx), -1)
noise_cov = mne.compute_raw_covariance(
    mne.io.RawArray(baseline_flat, info), method='empirical'
)

# Delete stale cache if it exists (recompute so the avg-ref projector is included)
if os.path.exists(INV_FILE):
    os.remove(INV_FILE)
    print('Deleted stale cached inverse — recomputing with avg-ref projector.')

inv = mne.minimum_norm.make_inverse_operator(
    info, fwd, noise_cov, loose=0.2, depth=0.8
)
mne.minimum_norm.write_inverse_operator(INV_FILE, inv)
print('Inverse operator computed and cached.')

## 4 — Apply source localization

Project both actual and predicted EEG grand averages onto the cortical surface.
We compute the sLORETA source estimate for each of the 200 test images and
store the time-averaged map over three visual windows.

In [ ]:
MAPS_FILE = f'{VIZ_CACHE}/source_maps.npz'

def ms_to_idx(ms_val):
    return int(np.argmin(np.abs(times_ms - ms_val)))

windows = {
    'P1 (80–140ms)':    (ms_to_idx(80),  ms_to_idx(140)),
    'N170 (130–200ms)': (ms_to_idx(130), ms_to_idx(200)),
    'P2 (200–300ms)':   (ms_to_idx(200), ms_to_idx(300)),
}

lambda2 = 1.0 / 9.0  # SNR=3

def eeg_to_source_maps(eeg_grand_avg, label=''):
    """eeg_grand_avg: (200, n_ch_keep, 100) → dict of (200, n_src) per window."""
    n_src = inv['src'][0]['nuse'] + inv['src'][1]['nuse']
    maps  = {w: np.zeros((200, n_src), dtype=np.float32) for w in windows}

    for img_idx in range(200):
        epoch = eeg_grand_avg[img_idx]       # (n_ch, 100)
        tmin  = times_ms[0] / 1000.0

        evoked = mne.EvokedArray(epoch, info, tmin=tmin, nave=1)
        evoked.apply_proj()                  # apply avg-ref projector

        stc = mne.minimum_norm.apply_inverse(
            evoked, inv, lambda2=lambda2, method='sLORETA', verbose=False
        )

        for win_name, (t0, t1) in windows.items():
            maps[win_name][img_idx] = stc.data[:, t0:t1+1].mean(axis=1)

        if img_idx % 50 == 0:
            print(f'  {label} {img_idx}/200 done', flush=True)

    return maps


if os.path.exists(MAPS_FILE):
    cache = np.load(MAPS_FILE)
    actual_maps  = {w: cache[f'actual_{i}'] for i, w in enumerate(windows)}
    pred_maps    = {w: cache[f'pred_{i}']   for i, w in enumerate(windows)}
    src_vertices = [cache['lh_verts'], cache['rh_verts']]
    print('Loaded cached source maps.')
else:
    print('Computing actual EEG source maps...')
    actual_maps = eeg_to_source_maps(grand_avg_actual[:, keep_idx, :], label='actual')
    print('Computing predicted EEG source maps...')
    pred_maps   = eeg_to_source_maps(grand_avg_pred[:, keep_idx, :],   label='pred')

    src_vertices = [inv['src'][0]['vertno'], inv['src'][1]['vertno']]
    save_dict = {f'actual_{i}': actual_maps[w] for i, w in enumerate(windows)}
    save_dict.update({f'pred_{i}': pred_maps[w] for i, w in enumerate(windows)})
    save_dict['lh_verts'] = src_vertices[0]
    save_dict['rh_verts'] = src_vertices[1]
    np.savez(MAPS_FILE, **save_dict)
    print('Source maps cached.')

window_names = list(windows.keys())
print('Done. Source map shapes:', list(actual_maps.values())[0].shape)

## 5 — Load surface geometry for Plotly

In [ ]:
import numpy as np

def get_src_mesh(hemi_src):
    """
    Extract source-space vertex coords (mm) and a valid triangulation.
    use_tris indices are in the FULL rr space — must remap to vertno-subset indices.
    """
    vertno = hemi_src['vertno']
    rr = hemi_src['rr'][vertno] * 1000  # m → mm, shape (n_src, 3)

    tris = None
    if 'use_tris' in hemi_src and hemi_src['use_tris'] is not None and len(hemi_src['use_tris']) > 0:
        # Remap from full-surface indices to subset indices
        full_to_sub = np.full(len(hemi_src['rr']), -1, dtype=np.int32)
        full_to_sub[vertno] = np.arange(len(vertno), dtype=np.int32)
        remapped = full_to_sub[hemi_src['use_tris']]
        valid = (remapped >= 0).all(axis=1)
        tris = remapped[valid].astype(np.int32)

    if tris is None or len(tris) == 0:
        from scipy.spatial import ConvexHull
        try:
            tris = ConvexHull(rr).simplices.astype(np.int32)
        except Exception:
            from scipy.spatial import Delaunay
            tris = Delaunay(rr[:, :2]).simplices.astype(np.int32)

    return rr.astype(np.float32), tris


lh_coords, lh_faces = get_src_mesh(inv['src'][0])
rh_coords, rh_faces = get_src_mesh(inv['src'][1])

# Offset RH in x so hemispheres don't overlap
x_gap = lh_coords[:, 0].max() - rh_coords[:, 0].min() + 20
rh_coords_off = rh_coords.copy()
rh_coords_off[:, 0] += x_gap

n_lh_verts = len(lh_coords)
rh_faces_off = rh_faces + n_lh_verts

all_coords = np.vstack([lh_coords, rh_coords_off])
all_faces  = np.vstack([lh_faces,  rh_faces_off])

print(f'Source mesh: {len(all_coords)} vertices, {len(all_faces)} faces')
print(f'Coord range  x:[{all_coords[:,0].min():.1f},{all_coords[:,0].max():.1f}]  '
      f'y:[{all_coords[:,1].min():.1f},{all_coords[:,1].max():.1f}]  '
      f'z:[{all_coords[:,2].min():.1f},{all_coords[:,2].max():.1f}]')

In [ ]:
# Source estimates are already at the source vertices — no mapping needed.
# Just concatenate LH and RH source values directly.

n_lh_src = inv['src'][0]['nuse']

def src_to_mesh(src_map_1d):
    """src_map_1d: (n_lh_src + n_rh_src,) already aligned to mesh vertices."""
    return src_map_1d.astype(np.float32)

surf_actual = {w: np.stack([src_to_mesh(actual_maps[w][i]) for i in range(200)]) for w in window_names}
surf_pred   = {w: np.stack([src_to_mesh(pred_maps[w][i])   for i in range(200)]) for w in window_names}

# Free large per-subject arrays to save RAM
del actual_eeg, pred_eeg
import gc; gc.collect()

print('Surface arrays ready:', list(surf_actual.values())[0].shape,
      f'  (~{list(surf_actual.values())[0].nbytes * 6 / 1e6:.1f} MB total for all variants)')

## 6 — Load test images for display

In [ ]:
import zipfile, os
from PIL import Image
import io, base64

# Find test_images.zip anywhere under ROOT
def find_file(root, filename):
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

img_zip_path = find_file(ROOT, 'test_images.zip')
if img_zip_path is None:
    raise FileNotFoundError(
        f"test_images.zip not found under {ROOT}. "
        "Check Drive and update IMG_ZIP manually if it's elsewhere."
    )
print('Found test_images.zip at:', img_zip_path)

def img_to_b64(pil_img, size=(120, 120)):
    pil_img = pil_img.convert('RGB').resize(size, Image.LANCZOS)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=80)
    return base64.b64encode(buf.getvalue()).decode()

print('Loading test images from zip...')
img_b64 = []
with zipfile.ZipFile(img_zip_path) as zf:
    img_names = sorted([n for n in zf.namelist() if n.lower().endswith(('.jpg', '.jpeg', '.png'))])
    for name in img_names[:200]:
        with zf.open(name) as f:
            img = Image.open(f)
            img_b64.append(img_to_b64(img))

print(f'Loaded {len(img_b64)} test images as base64 thumbnails.')

## 7 — Build interactive Plotly visualization

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_surf_vals(img_idx, condition, window):
    return surf_actual[window][img_idx] if condition == 'Actual' else surf_pred[window][img_idx]

def clim(vals):
    nz = vals[vals != 0]
    return float(np.percentile(np.abs(nz), 99)) if len(nz) > 0 else 1.0

# ── Diagnostics ───────────────────────────────────────────────────────────────
init_vals = get_surf_vals(0, 'Predicted', window_names[1])
print(f'init_vals  min={init_vals.min():.4e}  max={init_vals.max():.4e}  '
      f'NaN={np.isnan(init_vals).sum()}  nonzero={np.count_nonzero(init_vals)}')
print(f'all_coords shape: {all_coords.shape}  all_faces shape: {all_faces.shape}')
assert len(init_vals) == len(all_coords), \
    f'intensity length {len(init_vals)} != n_vertices {len(all_coords)}'

init_clim = clim(init_vals)

# ── FigureWidget (supports Python callbacks → no frame-sync problem) ──────────
fig = go.FigureWidget(data=[go.Mesh3d(
    x=all_coords[:, 0], y=all_coords[:, 1], z=all_coords[:, 2],
    i=all_faces[:, 0],  j=all_faces[:, 1],  k=all_faces[:, 2],
    intensity=init_vals,
    intensitymode='vertex',
    colorscale='RdBu_r',
    cmin=-init_clim, cmax=init_clim,
    colorbar=dict(title='sLORETA', thickness=15, len=0.6),
    lighting=dict(ambient=0.8, diffuse=0.6, specular=0.1, roughness=0.6),
    lightposition=dict(x=100, y=200, z=0),
    showscale=True,
)])

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        bgcolor='#111111',
        camera=dict(eye=dict(x=0, y=2.0, z=0.5)),
        aspectmode='data',
    ),
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    font=dict(color='white'),
    margin=dict(l=0, r=0, b=0, t=40),
    height=600,
)

# ── ipywidgets controls ───────────────────────────────────────────────────────
img_slider   = widgets.IntSlider(min=0, max=199, step=1, value=0,
                                  description='Image:', continuous_update=False,
                                  layout=widgets.Layout(width='70%'))
cond_toggle  = widgets.ToggleButtons(options=['Predicted', 'Actual'],
                                      description='Condition:',
                                      style={'button_width': '110px'})
win_toggle   = widgets.ToggleButtons(options=window_names,
                                      description='Window:',
                                      style={'button_width': '140px'})
title_label  = widgets.Label(value=f'Image 1/200 | Predicted | {window_names[1]}')

def update(_change=None):
    img_idx   = img_slider.value
    condition = cond_toggle.value
    window    = win_toggle.value
    vals = get_surf_vals(img_idx, condition, window)
    cl   = clim(vals)
    with fig.batch_update():
        fig.data[0].intensity = vals
        fig.data[0].cmin = -cl
        fig.data[0].cmax =  cl
    title_label.value = f'Image {img_idx+1}/200 | {condition} | {window}'

img_slider.observe(update,  names='value')
cond_toggle.observe(update, names='value')
win_toggle.observe(update,  names='value')

ui = widgets.VBox(
    [title_label, cond_toggle, win_toggle, img_slider, fig],
    layout=widgets.Layout(
        width='860px',
        background='#111111',
        padding='10px',
        border='1px solid #333',
    )
)
print('Controls built — run next cell to display.')

## 8 — Display

The cell below renders the figure inline.
If the Plotly widget is too large for Colab, use `fig.write_html('brain_viz.html')` to export
a standalone HTML file, then download and open it locally.

In [ ]:
display(ui)

In [ ]:
from IPython.display import HTML
display(HTML("""
<style>
  .widget-vbox { background: #111111 !important; }
  .jp-OutputArea-output { background: #111111 !important; }
  .widget-label { color: white !important; }
  .widget-toggle-buttons .widget-toggle-button {
      background: #333 !important;
      color: white !important;
      border-color: #555 !important;
  }
  .widget-toggle-buttons .widget-toggle-button.mod-active {
      background: #555 !important;
  }
</style>
"""))

In [ ]:
# Export standalone HTML (download and open in browser for best experience)
out_html = f'{WRITE_ROOT}/brain_viz_interactive.html'
fig.write_html(out_html, include_plotlyjs='cdn')
print('Exported to', out_html)

## Bonus — Side-by-side image + ERP panel

Show the stimulus image alongside the grand-average ERP waveform
for a chosen image and the occipital channels.

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display
import matplotlib.pyplot as plt

OCC_CHANNELS = ['Oz', 'O1', 'O2', 'POz', 'PO7', 'PO8']
occ_idx = [eeg_ch_names.index(c) for c in OCC_CHANNELS if c in eeg_ch_names]

def show_erp(img_idx):
    fig2, axes = plt.subplots(1, 2, figsize=(12, 4),
                              gridspec_kw={'width_ratios': [1, 3]})

    # Left: stimulus image
    from PIL import Image as PILImage
    img_bytes = base64.b64decode(img_b64[img_idx])
    img = PILImage.open(io.BytesIO(img_bytes))
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title(f'Test image {img_idx+1}')

    # Right: ERP waveforms — occipital channels, actual vs predicted
    act  = grand_avg_actual[img_idx][occ_idx].mean(axis=0)  # (100,)
    pred = grand_avg_pred[img_idx][occ_idx].mean(axis=0)    # (100,)
    axes[1].plot(times_ms, act,  label='Actual',    color='steelblue', lw=2)
    axes[1].plot(times_ms, pred, label='Predicted', color='tomato',    lw=2, ls='--')
    axes[1].axvline(0, color='gray', ls=':')
    axes[1].axvspan(80, 300, alpha=0.1, color='green', label='Visual window')
    axes[1].set_xlabel('Time (ms)')
    axes[1].set_ylabel('Amplitude (µV)')
    axes[1].set_title('Occipital grand-avg ERP (actual vs predicted)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

slider = widgets.IntSlider(value=0, min=0, max=199, step=1, description='Image:')
widgets.interact(show_erp, img_idx=slider);